In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

data_dir = r"D:\ImageNet"
img_size = 64
batch_size = 32
latent_dim = 100
embedding_dim = 128

ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, image_size=(img_size, img_size),
    batch_size=batch_size, label_mode="int"
)

class_names = ds.class_names
captions = ["a photo of a " + c.replace("_", " ").lower() for c in class_names]

tokenizer = tf.keras.layers.TextVectorization(
    output_mode="int", output_sequence_length=10
)
tokenizer.adapt(tf.data.Dataset.from_tensor_slices(captions))
vocab_size = len(tokenizer.get_vocabulary())

text_input = tf.keras.Input(shape=(10,))
x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(text_input)
text_encoder = tf.keras.Model(text_input, tf.keras.layers.GlobalAveragePooling1D()(x))

noise_input = tf.keras.Input(shape=(latent_dim,))
condition_input = tf.keras.Input(shape=(embedding_dim,))
x = tf.keras.layers.Concatenate()([noise_input, condition_input])
x = tf.keras.layers.Dense(8 * 8 * 256, activation="relu")(x)
x = tf.keras.layers.Reshape((8, 8, 256))(x)
for filters in [128, 64]:
    x = tf.keras.layers.Conv2DTranspose(filters, 4, strides=2, padding="same", activation="relu")(x)
generated = tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh")(x)
generator = tf.keras.Model([noise_input, condition_input], generated)

image_input = tf.keras.Input(shape=(img_size, img_size, 3))
disc_text_input = tf.keras.Input(shape=(embedding_dim,))
x = tf.keras.layers.Conv2D(64, 4, strides=2, padding="same")(image_input)
x = tf.keras.layers.LeakyReLU(0.2)(x)
x = tf.keras.layers.Conv2D(128, 4, strides=2, padding="same")(x)
x = tf.keras.layers.LeakyReLU(0.2)(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Concatenate()([x, disc_text_input])
x = tf.keras.layers.Dense(256, activation="leaky_relu")(x)
validity = tf.keras.layers.Dense(1, activation="sigmoid")(x)
discriminator = tf.keras.Model([image_input, disc_text_input], validity)
discriminator.compile(optimizer="adam", loss="binary_crossentropy")

discriminator.trainable = False
z = tf.keras.Input(shape=(latent_dim,))
cond = tf.keras.Input(shape=(embedding_dim,))
fake = generator([z, cond])
valid = discriminator([fake, cond])
gan = tf.keras.Model([z, cond], valid)
gan.compile(optimizer="adam", loss="binary_crossentropy")

for epoch in range(10):
    for real_images, labels in ds:
        real_images = (tf.cast(real_images, tf.float32) / 127.5) - 1
        text_vectors = text_encoder(tokenizer(captions)[labels])
        n = tf.shape(real_images)[0]
        noise = tf.random.normal((n, latent_dim))
        fake_images = generator([noise, text_vectors], training=True)

        discriminator.trainable = True
        discriminator.train_on_batch([real_images, text_vectors], tf.ones((n, 1)))
        discriminator.train_on_batch([fake_images, text_vectors], tf.zeros((n, 1)))

        discriminator.trainable = False
        gan.train_on_batch([noise, text_vectors], tf.ones((n, 1)))

text = "a photo of a tiger"
text_vector = text_encoder(tokenizer([text]))
noise = tf.random.normal((1, latent_dim))
generated_image = generator.predict([noise, text_vector], verbose=0)[0]
generated_image = (generated_image + 1) / 2

plt.imshow(np.clip(generated_image, 0, 1))
plt.title(text)
plt.axis("off")
plt.show()

In [ ]:

import os
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

data_dir = r"D:\ImageNet\Images"
img_size = 64
batch_size = 32
latent_dim = 100
embedding_dim = 128

classes = sorted([d for d in os.listdir(data_dir)
                  if os.path.isdir(os.path.join(data_dir, d))])[:5]

ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, class_names=classes, image_size=(img_size, img_size),
    batch_size=batch_size, label_mode="int"
)

captions = ["a photo of a " + c.replace("_", " ").lower() for c in classes]

tokenizer = tf.keras.layers.TextVectorization(output_mode="int", output_sequence_length=10)
tokenizer.adapt(tf.data.Dataset.from_tensor_slices(captions))
vocab_size = len(tokenizer.get_vocabulary())

text_input = tf.keras.Input(shape=(10,))
x = tf.keras.layers.Embedding(vocab_size, embedding_dim)(text_input)
text_encoder = tf.keras.Model(text_input, tf.keras.layers.GlobalAveragePooling1D()(x))

noise_input = tf.keras.Input(shape=(latent_dim,))
condition_input = tf.keras.Input(shape=(embedding_dim,))
x = tf.keras.layers.Concatenate()([noise_input, condition_input])
x = tf.keras.layers.Dense(8 * 8 * 256, activation="relu")(x)
x = tf.keras.layers.Reshape((8, 8, 256))(x)
for filters in [128, 64]:
    x = tf.keras.layers.Conv2DTranspose(filters, 4, strides=2, padding="same", activation="relu")(x)
generated = tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh")(x)
generator = tf.keras.Model([noise_input, condition_input], generated)

image_input = tf.keras.Input(shape=(img_size, img_size, 3))
disc_text_input = tf.keras.Input(shape=(embedding_dim,))
x = tf.keras.layers.Conv2D(64, 4, strides=2, padding="same")(image_input)
x = tf.keras.layers.LeakyReLU(0.2)(x)
x = tf.keras.layers.Conv2D(128, 4, strides=2, padding="same")(x)
x = tf.keras.layers.LeakyReLU(0.2)(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Concatenate()([x, disc_text_input])
x = tf.keras.layers.Dense(256)(x)
x = tf.keras.layers.LeakyReLU(0.2)(x)
validity = tf.keras.layers.Dense(1, activation="sigmoid")(x)
discriminator = tf.keras.Model([image_input, disc_text_input], validity)

discriminator.compile(optimizer="adam", loss="binary_crossentropy")
discriminator.trainable = False

z = tf.keras.Input(shape=(latent_dim,))
cond = tf.keras.Input(shape=(embedding_dim,))
fake = generator([z, cond])
valid = discriminator([fake, cond])
gan = tf.keras.Model([z, cond], valid)
gan.compile(optimizer="adam", loss="binary_crossentropy")

discriminator.trainable = True

for epoch in range(5):
    losses = []
    for real_images, labels in ds.take(100):
        real_images = tf.cast(real_images, tf.float32) / 127.5 - 1
        text_vectors = text_encoder(tokenizer(tf.gather(captions, labels)))
        n = tf.shape(real_images)[0]
        noise = tf.random.normal((n, latent_dim))
        fake_images = generator([noise, text_vectors], training=False)

        d_real = discriminator.train_on_batch([real_images, text_vectors], tf.ones((n, 1)))
        d_fake = discriminator.train_on_batch([fake_images, text_vectors], tf.zeros((n, 1)))
        discriminator.trainable = False
        g_loss = gan.train_on_batch([noise, text_vectors], tf.ones((n, 1)))
        discriminator.trainable = True
        losses.append(float(g_loss))

    print(f"Epoch {epoch+1}/5, Generator loss: {np.mean(losses):.4f}")

text = captions[0]
text_vector = text_encoder(tokenizer([text]))
noise = tf.random.normal((1, latent_dim))
generated_image = generator.predict([noise, text_vector], verbose=0)[0]

plt.imshow(np.clip((generated_image + 1) / 2, 0, 1))
plt.title(text)
plt.axis("off")
plt.show()
